# 06 - Validate the estimated CEFR bands against Kelly

Phase 6, step 8. Kelly (en/it) is **CC-BY-NC-SA and validation-only**: nothing
here writes to the corpus and no graded label is ever merged into a shipped
table. Spec: [`06_frequency_complexity.md`](../../scratch_space/09_concept_model/06_frequency_complexity.md).

Thin caller over `ingestion.cefr_validation`. Needs the staged Kelly lists
(`00_stage.ipynb`, or `download_cefr_source`) and the `store` extra.

Two things this notebook can and cannot settle:

- it **can** validate the score's *ordering* (rank correlation), and the Italian
  number is the interesting one because the cutoffs were fitted on English alone;
- it **cannot** set the absolute scale - Kelly grades the ~7.5k most frequent
  words, so its own "C2" means "least frequent of the common head", not "hardest
  word in the language".

Circularity caveat (5.54): Kelly's bands were themselves built largely from
corpus frequency, the heaviest term in our score, so agreement shows the pipeline
is internally consistent, not that it predicts human-perceived difficulty.

In [ ]:
from loguru import logger as lg

from lang_tools.lexicon.ingestion.cefr_validation import band_agreement
from lang_tools.lexicon.ingestion.cefr_validation import corpus_form_scores
from lang_tools.lexicon.ingestion.cefr_validation import fit_cutoffs
from lang_tools.lexicon.ingestion.cefr_validation import graded_pairs
from lang_tools.lexicon.ingestion.cefr_validation import rank_correlation
from lang_tools.lexicon.ingestion.cefr_validation import staged_graded_list
from lang_tools.lexicon.ingestion.enrich import CEFR_CUTOFFS
from lang_tools.params.lang_tools_params import get_lang_tools_params

GRADED_LANGS = ["en", "it"]  # the only two Kelly covers
data_fol = get_lang_tools_params().paths.data_fol
lg.info("06-validation: langs={} -> data_fol={}", GRADED_LANGS, data_fol)
data_fol

## Join the corpus scores to the graded list

The score is not persisted (only the band it produces is), so it is recomputed
from the same inputs the build used. A form's score is the *easiest* of its
senses, because a graded list grades the word as a learner meets it.

In [ ]:
pairs = {}
for lang in GRADED_LANGS:
    scores = corpus_form_scores(data_fol, lang)
    graded = staged_graded_list(data_fol, lang)
    pairs[lang] = graded_pairs(scores, graded)
    lg.info(
        "{}: {} corpus forms, {} graded, {} matched",
        lang, len(scores), len(graded), len(pairs[lang]),
    )

## Agreement with the shipped cutoffs

`exact` and `within_one` are band matches; `mean_offset` is signed - negative
means we call words easier than Kelly does.

In [ ]:
for lang in GRADED_LANGS:
    agreement = band_agreement(pairs[lang], CEFR_CUTOFFS)
    lg.info("{}: {}", lang, {k: round(v, 3) for k, v in agreement.items()})

## The ordering: what Kelly can actually support

Rank correlation between our score and Kelly's bands. English is the language
the cutoffs were fitted on; **Italian is the real test**, since the concept-level
part of the score is supposed to travel across languages (5.54 Topic 5).

In [ ]:
for lang in GRADED_LANGS:
    lg.info("{}: spearman(score, kelly band) = {:.3f}", lang, rank_correlation(pairs[lang]))

## Re-fit the cutoffs

Run this when the score changes: it re-derives `enrich.CEFR_CUTOFFS` by matching
Kelly's band proportions on the English subset. Compare the agreement before
adopting - and re-run the build, since the bands are written by it.

In [ ]:
fitted = fit_cutoffs(pairs["en"])
lg.info("shipped: {}", CEFR_CUTOFFS)
lg.info("fitted:  {}", tuple(round(c, 4) for c in fitted))
lg.info("shipped agreement: {}", {k: round(v, 3) for k, v in band_agreement(pairs["en"], CEFR_CUTOFFS).items()})
lg.info("fitted agreement:  {}", {k: round(v, 3) for k, v in band_agreement(pairs["en"], fitted).items()})